<link rel="stylesheet" href="/site-assets/css/gemma.css">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Google+Symbols:opsz,wght,FILL,GRAD@20..48,100..700,0..1,-50..200" />

##### Copyright 2025 Google LLC。

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table class="tfo-notebook-buttons" align="left"> <td>    <a target="_blank" href="https://ai.google.dev/gemma/docs/core/pytorch_gemma"><img src="https://ai.google.dev/static/site-assets/images/docs/notebook-site-button.png" height="32" width="32" />View on ai.google.dev</a>
</td> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/docs/core/pytorch_gemma.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td> <td>    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/google-gemini/gemma-cookbook/blob/main/docs/core/pytorch_gemma.ipynb"><img src="https://www.kaggle.com/static/images/logos/kaggle-logo-transparent-300.png" height="32" width="70"/>Run in Kaggle</a>
</td> <td>    <a target="_blank" href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgoogle-gemini%2Fgemma-cookbook%2Fmain%2Fdocs%2Fcore%2Fpytorch_gemma.ipynb"><img src="https://ai.google.dev/images/cloud-icon.svg" width="40" />Open in Vertex AI</a>
</td> <td>    <a target="_blank" href="https://github.com/google-gemini/gemma-cookbook/blob/main/docs/core/pytorch_gemma.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
</td>
</table>

# 使用PyTorch執行Gemma

本指南向您展示如何使用 PyTorch framework 執行 Gemma，包括如何
將影像資料用於 prompting Gemma 版本 3 及更高版本的型號。了解更多
有關 Gemma PyTorch 實施的詳細信息，請參閱項目存儲庫
[自述文件](https://github.com/google/gemma_pytorch)。

## 設定

以下部分說明如何設定您的開發環境，
包括如何存取Gemma模型以從Kaggle下載，設置
身份驗證變數、安裝相依性和導入套件。

### 系統需求

這Gemma Pytorch library需要 GPU 或 TPU 處理器來運作Gemma
模型。標準 Colab CPU Python runtime 和 T4 GPU Python runtime 是
足以執行 Gemma 1B、2B 和 4B 尺寸型號。對於進階用例
其他 GPU 或 TPU，請參閱
[自述文件](https://github.com/google/gemma_pytorch/blob/main/README.md) 在
Gemma PyTorch 倉庫。

### 造訪 Kaggle 上的 Gemma

要完成本教學，您首先需要按照以下位置的設定說明進行操作
[Gemma 設定](https://ai.google.dev/gemma/docs/setup)，它向您展示如何操作
以下：
* 造訪 [Kaggle](https://www.kaggle.com/models/google/gemma/) 上的 Gemma。
* 選擇具有足夠資源的 Colab runtime 來執行 Gemma 模型。
* 產生並設定 Kaggle 使用者名稱和 API 金鑰。

完成 Gemma 設定後，請繼續下一部分，其中
您將為Colab 環境設定環境變數。

### 設定環境變數

設定`KAGGLE_USERNAME` 和`KAGGLE_KEY` 的環境變數。當prompted 時
與「授予存取權限？」訊息，同意提供secret存取。

In [ ]:
import os
from google.colab import userdata # `userdata` is a Colab API.

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

### 安裝依賴項

In [ ]:
!pip install -q -U torch immutabledict sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.4/209.4 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 55.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.7.15 requires torch<2.4,>=1.10, but you have torch 2.4.0 which is incompatible.
torchaudio 2.3.1+cu121 requires torch==2.3.1, but you have torch 2.4.0 which is incompatible.
torchvision 0.18.1+cu121 requires torch==2.3.1, but you have torch 2.4.0 which is incompatible.


### 下載模型權重

In [ ]:
# Choose variant and machine type
VARIANT = '4b-it'  #@param ['1b','1b-it','4b','4b-it','12b','12b-it','27b','27b-it']
MACHINE_TYPE = 'cuda' #@param ['cuda', 'cpu']
CONFIG = VARIANT.split('-')[0]

In [ ]:
import kagglehub

# Load model weights
weights_dir = kagglehub.model_download(f'google/gemma-3/pyTorch/gemma-3-{VARIANT}')

設定模型的 tokenizer 和 checkpoint 路徑。

In [ ]:
# Ensure that the tokenizer is present
tokenizer_path = os.path.join(weights_dir, 'tokenizer.model')
assert os.path.isfile(tokenizer_path), 'Tokenizer not found!'

# Ensure that the checkpoint is present
ckpt_path = os.path.join(weights_dir, f'model.ckpt')
assert os.path.isfile(ckpt_path), 'PyTorch checkpoint not found!'

## 設定運作環境

以下各節介紹如何準備 PyTorch 執行環境
Gemma。

### 準備PyTorch執行環境

透過克隆Gemma Pytorch 來準備PyTorch模型執行環境
儲存庫。

In [ ]:
!git clone https://github.com/google/gemma_pytorch.git

Cloning into 'gemma_pytorch'...
remote: Enumerating objects: 239, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 239 (delta 86), reused 58 (delta 55), pack-reused 116
Receiving objects: 100% (239/239), 2.18 MiB | 20.83 MiB/s, done.
Resolving deltas: 100% (135/135), done.


In [ ]:
import sys

sys.path.append('gemma_pytorch/gemma')

In [ ]:
from gemma_pytorch.gemma.config import get_model_config
from gemma_pytorch.gemma.gemma3_model import Gemma3ForMultimodalLM

import os
import torch

### 設定模型設定

在執行模型之前，您必須設定一些設定參數，包括
Gemma 變體、tokenizer 和量化等級。

In [ ]:
# Set up model config.
model_config = get_model_config(CONFIG)
model_config.dtype = "float32" if MACHINE_TYPE == "cpu" else "float16"
model_config.tokenizer = tokenizer_path

### 設定設備上下文

以下程式碼設定用於執行模型的設備上下文：

In [ ]:
@contextlib.contextmanager
def _set_default_tensor_type(dtype: torch.dtype):
    """Sets the default torch dtype to the given dtype."""
    torch.set_default_dtype(dtype)
    yield
    torch.set_default_dtype(torch.float)

### 實例化並載入模型

載入模型及其權重以準備執行請求。

In [ ]:
device = torch.device(MACHINE_TYPE)
with _set_default_tensor_type(model_config.get_dtype()):
    model = Gemma3ForMultimodalLM(model_config)
    model.load_state_dict(torch.load(ckpt_path)['model_state_dict'])
    model = model.to(device).eval()
print("Model loading done.")

print('Generating requests in chat mode...')

## 執行inference

以下是聊天模式產生和多生成的範例
請求。
經過指令調整的 Gemma 模型使用特定的格式化程序進行訓練，
使用額外資訊註釋指令調整範例，兩者皆在
培訓和inference。註 (1) 表示對話中的角色，
(2) 劃定對話的輪次。
相關註tokens為：
- `user`：使用者輪流
- `model`：模型轉動
- `<start_of_turn>`：對話開始
- `<start_of_image>`：影像資料輸入標籤
- `<end_of_turn><eos>`：對話結束

有關詳細信息，請閱讀 prompt 指令調整格式 Gemma
模型[此處](https://ai.google.dev/gemma/core/prompt-structure)。

### 用文本生成文本

以下是範例程式碼片段，示範如何格式化prompt
使用使用者和模型聊天範本進行指令調整的 Gemma 模型
多輪對話。

In [ ]:
# Chat templates
USER_CHAT_TEMPLATE = "<start_of_turn>user\n{prompt}<end_of_turn><eos>\n"
MODEL_CHAT_TEMPLATE = "<start_of_turn>model\n{prompt}<end_of_turn><eos>\n"

# Sample formatted prompt
prompt = (
    USER_CHAT_TEMPLATE.format(
        prompt='What is a good place for travel in the US?'
    )
    + MODEL_CHAT_TEMPLATE.format(prompt='California.')
    + USER_CHAT_TEMPLATE.format(prompt='What can I do in California?')
    + '<start_of_turn>model\n'
)
print('Chat prompt:\n', prompt)

model.generate(
    USER_CHAT_TEMPLATE.format(prompt=prompt),
    device=device,
    output_len=256,
)

Chat prompt:
 <start_of_turn>user
What is a good place for travel in the US?<end_of_turn><eos>
<start_of_turn>model
California.<end_of_turn><eos>
<start_of_turn>user
What can I do in California?<end_of_turn><eos>
<start_of_turn>model



"California is a state brimming with diverse activities! To give you a great list, tell me: \n\n* **What kind of trip are you looking for?** Nature, City life, Beach, Theme Parks, Food, History, something else? \n* **What are you interested in (e.g., hiking, museums, art, nightlife, shopping)?** \n* **What's your budget like?** \n* **Who are you traveling with?** (family, friends, solo)  \n\nThe more you tell me, the better recommendations I can give! 😊  \n<end_of_turn>"

In [ ]:
# Generate sample
model.generate(
    'Write a poem about an llm writing a poem.',
    device=device,
    output_len=100,
)

"\n\nA swirling cloud of data, raw and bold,\nIt hums and whispers, a story untold.\nAn LLM whispers, code into refrain,\nCrafting words of rhyme, a lyrical strain.\n\nA world of pixels, logic's vibrant hue,\nFlows through its veins, forever anew.\nThe human touch it seeks, a gentle hand,\nTo mold and shape, understand.\n\nEmotions it might learn, from snippets of prose,\nInspiration it seeks, a yearning"

### 用圖像生成文字

對於 Gemma 版本 3 及更高版本，您可以將圖像與 prompt 一起使用。的
以下範例向您展示如何將視覺資料包含在 prompt 中。

In [ ]:
print('Chat with images...\n')

def read_image(url):
    import io
    import requests
    import PIL

    contents = io.BytesIO(requests.get(url).content)
    return PIL.Image.open(contents)

image = read_image(
    'https://storage.googleapis.com/keras-cv/models/paligemma/cow_beach_1.png'
)

print(model.generate(
    [
        [
            '<start_of_turn>user\n',
            image,
            'What animal is in this image?<end_of_turn>\n',
            '<start_of_turn>model\n'
        ]
    ],
    device=device,
    output_len=256,
))

## 了解更多

現在您已經學會如何在 Pytorch 中使用Gemma，您可以探索許多
Gemma 可以做的其他事情
[ai.google.dev/gemma](https://ai.google.dev/gemma)。
另請參閱這些其他相關資源：
- [Gemma核心模型概述](https://ai.google.dev/gemma/docs/core)
- [Gemma C++ 教學](https://ai.google.dev/gemma/docs/core/gemma_cpp)
- [Gemma prompt 與系統指令](https://ai.google.dev/gemma/core/prompt-structure)